# README figures

Generates the three figures embedded in the repository README. Nothing here is new
analysis — each panel re-draws a result already computed in [`EDA.ipynb`](EDA.ipynb),
and prints the numbers the README captions quote so the two can be checked against
each other.

Run from inside `notebooks/` (the data path is relative to the working directory):

```sh
cd notebooks && uv run jupyter lab
```

then *Run All*. The PNG files land in `../figures/` and are committed, so a reader
without the workbook still sees them. Every number printed below is the same value as
in the corresponding `EDA.ipynb` section; if one drifts, the figure is stale.

| figure | source section |
| --- | --- |
| `01_units_vs_spend.png` | EDA §2, weekly totals and trend |
| `02_promotion_economics.png` | EDA §6.2, discount against incremental spend |
| `03_basket_economics.png` | EDA §6.3, spend per basket and units per basket |

Lift is reported as a **ratio** (1.48x), not a percentage change (+47.7%): the two are
easy to confuse, and `EDA.ipynb` §6.2 labels the same ratio as `+147.7%`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 4.5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

DATA = Path("../data/dunnhumby - Breakfast at the Frat.xlsx")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 50)

# palette carried over from 1_distribution_analysis.ipynb
PROMO_COL = "#e8b33a"
RED, BLUE = "#c44e52", "#4c72b0"

print(f"figures -> {FIG.resolve()}")

## 1. Volume down, value up

Both series are indexed to their first week so they share one axis. The faint lines are
the raw weeks, the bold dashed lines the fitted trend, and the shaded wedge is the gap
between them — the point is the divergence, not the level.

In [ ]:
tx = pd.read_excel(DATA, "dh Transaction Data", header=1)

weekly = (tx.groupby("WEEK_END_DATE")
            .agg(UNITS=("UNITS", "sum"), SPEND=("SPEND", "sum"))
            .asfreq("W-WED"))
t = np.arange(len(weekly))

stats = {}
for col in ["UNITS", "SPEND"]:
    slope, intercept = np.polyfit(t, weekly[col], 1)
    stats[col] = (slope, slope * 52 / weekly[col].mean())
    print(f"{col:6s} slope {slope:>9,.1f}/wk = {stats[col][1]:+.2%}/yr")

fig, ax = plt.subplots()
trend = {}
for col, colour in [("UNITS", BLUE), ("SPEND", RED)]:
    idx = weekly[col] / weekly[col].iloc[0] * 100
    slope, intercept = np.polyfit(t, idx, 1)
    trend[col] = intercept + slope * t
    ax.plot(weekly.index, idx, color=colour, lw=0.8, alpha=0.30)
    ax.plot(weekly.index, trend[col], color=colour, lw=2.2,
            label=f"{col} trend  {stats[col][1]:+.2%}/yr")

ax.fill_between(weekly.index, trend["UNITS"], trend["SPEND"], color="gray", alpha=0.12)
ax.axhline(100, color="gray", lw=1, ls=":")
ax.set_ylabel("index, first week = 100")
ax.set_title("Weekly units and spend over 156 weeks: volume drifts down while value drifts up")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncols=2, frameon=False, fontsize=10)
fig.savefig(FIG / "01_units_vs_spend.png")
plt.show()

## 2. Promotion economics

The ratio the README quotes: discount given up against incremental spend, computed
exactly as [`EDA.ipynb`](EDA.ipynb) §6.2 does — a median paired SKU–store spend ratio
applied to each promo row's own non-promo baseline.

> The dataset has no cost or margin field, so this is revenue against discount. It is
> **not** a profit statement, and the chart says so on the axis.

In [ ]:
tx["ANY_PROMO"] = ((tx.DISPLAY == 1) | (tx.FEATURE == 1) | (tx.TPR_ONLY == 1)).astype(int)

pr = tx[(tx.PRICE > 0) & (tx.BASE_PRICE > 0)].copy()
pr["DISCOUNT"] = (pr.BASE_PRICE - pr.PRICE).clip(lower=0)
pr["DISC_SPEND"] = pr.DISCOUNT * pr.UNITS

cell = (pr.groupby(["UPC", "STORE_NUM", "ANY_PROMO"])
          .agg(units=("UNITS", "mean"), spend=("SPEND", "mean"), n=("UNITS", "size"))
          .unstack("ANY_PROMO"))
usable = (cell[("units", 0)] > 0) & (cell[("n", 0)] >= 4) & (cell[("n", 1)] >= 1)
unit_ratio = cell[("units", 1)][usable] / cell[("units", 0)][usable]
spend_ratio = cell[("spend", 1)][usable] / cell[("spend", 0)][usable]

baseline = cell[("spend", 0)][usable]
baseline = baseline[baseline > 0]
promo_rows = pr[pr.ANY_PROMO == 1].set_index(["UPC", "STORE_NUM"])
credit = pd.Series(promo_rows.index.map(baseline), index=promo_rows.index)
matched = credit.notna()

discount_cost = pr.loc[pr.ANY_PROMO == 1, "DISC_SPEND"].sum()
incr_spend = (credit[matched] * (spend_ratio.median() - 1)).sum()
ratio = incr_spend / discount_cost

print(f"usable cells: {usable.sum():,} of {len(usable):,}")
print(f"median paired ratio, units: {unit_ratio.median():.2f}x")
print(f"median paired ratio, spend: {spend_ratio.median():.2f}x")
print(f"discount given up:        ${discount_cost:>14,.0f}")
print(f"incremental spend:        ${incr_spend:>14,.0f}")
print(f"ratio:                              {ratio:.2f}x")

fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(["Discount given up\n(revenue forgone)", "Incremental spend\n(revenue gained)"],
              [discount_cost, incr_spend], color=[RED, PROMO_COL], width=0.55)
for b, v in zip(bars, [discount_cost, incr_spend]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"${v/1e6:,.2f}M",
            ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylim(0, max(discount_cost, incr_spend) * 1.12)
ax.set_yticks(np.arange(0, max(discount_cost, incr_spend) * 1.15, 1_000_000))
ax.yaxis.set_major_formatter(lambda x, _: f"${x/1e6:,.0f}M")
ax.set_title(f"Revenue return to promotion: {ratio:.2f}x")
ax.set_xlabel("no cost or margin field exists in this dataset — revenue against discount, not profit",
              fontsize=9, style="italic", color="dimgray")
fig.savefig(FIG / "02_promotion_economics.png")
plt.show()

## 3. Baskets: the receipt grows, the basket does not

Store-week aggregates, medians by ISO year. 2012 is a single week and is excluded —
see `EDA.ipynb` §2. The y-axes are zoomed so the direction is visible; the exact change
over the panel is printed above each panel.

In [ ]:
sw = (tx.groupby(["STORE_NUM", "WEEK_END_DATE"])
        .agg(spend=("SPEND", "sum"), units=("UNITS", "sum"),
             baskets=("VISITS", "sum"))
        .reset_index())
sw["per_basket"] = sw.spend / sw.baskets
sw["units_per_basket"] = sw.units / sw.baskets
sw["ISO_YEAR"] = sw.WEEK_END_DATE.dt.isocalendar().year.astype(int)

by_year = sw[sw.ISO_YEAR < 2012].groupby("ISO_YEAR")[["per_basket", "units_per_basket"]].median()
print(by_year.round(2).T)
print("change 2011 vs 2009: "
      f"spend/basket {(by_year.per_basket.iloc[-1] / by_year.per_basket.iloc[0] - 1):+.1%}, "
      f"units/basket {(by_year.units_per_basket.iloc[-1] / by_year.units_per_basket.iloc[0] - 1):+.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, col, colour, title in [
        (axes[0], "per_basket", RED, "Spend per basket"),
        (axes[1], "units_per_basket", BLUE, "Units per basket")]:
    change = by_year[col].iloc[-1] / by_year[col].iloc[0] - 1
    fmt = "{:,.2f}" if col == "per_basket" else "{:,.3f}"
    ax.plot(by_year.index, by_year[col], marker="o", color=colour, lw=2, ms=7)
    for x, y in by_year[col].items():
        ax.annotate(fmt.format(y), (x, y), textcoords="offset points", xytext=(0, 10),
                    ha="center", fontsize=10, fontweight="bold", color=colour)
    ax.set_xticks(by_year.index)
    ax.set_xlabel("ISO year")
    ax.set_title(f"{title} — median store-week, {change:+.1%} over the panel")
    ax.margins(y=0.20)

fig.suptitle("The receipt grows while the basket shrinks", y=1.03, fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "03_basket_economics.png")
plt.show()

## Regenerating

*Run All* rewrites the three PNGs in `../figures/`. The notebook is committed without
outputs; the figures are committed as files, because GitHub does not render notebook
outputs inside a README.

If a number printed here disagrees with the section of `EDA.ipynb` named in the table
at the top, the figure is out of date — fix whichever side is wrong, do not edit the
caption to match.